# ADPr-LLaMA Evaluation Notebook

`adpr-llama` is fine-tuned on positives-only data, following the precedent established by prior state-of-the-art ADPr-site predictors. The model consequently predicts at least one site in almost every 21-residue window. The complete inference architecture compensates for this: overlapping 21-residue windows (stride 5) are aggregated into a per-residue consensus score, and a single F1-optimal threshold derived from the per-residue ROC is applied as the decision rule.

## Calibration and test sets

Deriving an operating threshold from the same data on which the headline metrics are reported produces optimistically biased estimates. Two disjoint held-out sets are therefore used:

1. **`datasets/calibration.csv`** — 20% of the windowed training data, held out and unfiltered (containing both `has_ptm == 0` and `has_ptm == 1` rows). The overlapping-window consensus algorithm is run on this set and the F1-optimal threshold over the per-residue ROC is selected. Because the file is unfiltered, each calibration protein is fully tiled by stride-5 windows and the per-residue class distribution preserves the natural imbalance — a realistic mix of easy far-from-site negatives and hard near-site negatives. This distribution supports threshold generalization to unseen test proteins.
2. **`datasets/test.csv`** — unseen full proteins held out from training. Sliding-window inference is run, the calibration-derived threshold is applied without modification, and the final point metrics are reported: Accuracy, Precision, Recall (sensitivity), Specificity, F1, and the confusion matrix.

The selected threshold and windowing parameters are persisted to the model repository as `inference_config.json` for use by downstream inference code.

## 1. Environment Setup

In [ ]:
!pip install -q -U \
    "transformers>=4.44,<4.50" \
    "peft>=0.11" \
    "accelerate>=0.30" \
    "huggingface_hub>=0.24" \
    scikit-learn matplotlib seaborn "pandas<3" sentencepiece

# peft >=0.11 errors out if torchao is installed at an incompatible version.
# Colab ships torchao==0.10.0 pre-installed; we don't use it, so remove it.
!pip uninstall -y -q torchao

In [ ]:
import os, re, json, math, gc, time
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi, login
from sklearn.metrics import (
    roc_curve, roc_auc_score, confusion_matrix,
    precision_recall_fscore_support, accuracy_score,
)

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Configuration

In [ ]:
HF_REPO_ID = 'jbenbudd/adpr-llama'
CALIBRATION_CSV = 'datasets/calibration.csv'
TEST_CSV = 'datasets/test.csv'

WINDOW_SIZE = 21
STRIDE = 5
MAX_NEW_TOKENS = 32
BATCH_SIZE = 16

INSTRUCTION = '[Predict the ADP Ribosylation sites given the peptide sequence]'
ADPR_RESIDUES = ['D', 'E', 'R', 'K', 'S']

EVAL_DIR = 'eval_outputs'
os.makedirs(EVAL_DIR, exist_ok=True)

PUSH_MODEL_CARD = True

In [ ]:
try:
    from google.colab import userdata, drive
    HF_TOKEN = userdata.get('HF_API_TOKEN')
    # drive.mount('/content/drive', force_remount=False)
    # CALIBRATION_CSV = '/content/drive/MyDrive/adpr-llama/datasets/calibration.csv'
    # TEST_CSV = '/content/drive/MyDrive/adpr-llama/datasets/test.csv'
except ImportError:
    HF_TOKEN = os.environ.get('HF_API_TOKEN')

assert HF_TOKEN, 'No HF token found. Set HF_API_TOKEN in Colab Secrets or env vars.'
login(token=HF_TOKEN)
print('Logged in to Hugging Face.')

## 3. Load Model + Tokenizer from Hugging Face

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
model.config.use_cache = True
print('Loaded', HF_REPO_ID)

## 4. Sliding-Window Inference Helpers

Four helper functions used by both the calibration and test sections:

- `make_windows(seq)` — produces sliding-window starts (stride 5) with a tail window so that the final residues are covered.
- `build_prompt(window_seq)` — formats a 21-residue window into the Alpaca prompt used during training.
- `generate_batch(prompts)` — batched greedy generation; returns only the newly generated completions.
- `extract_sites(text)` — parses `Sites=<R5,D12,...>` from a completion and returns `[(letter, window_position), ...]`.

In [ ]:
PROMPT_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n{input}\n\n'
    '### Response:\n'
)

SITE_RE  = re.compile(r'^([A-Z])(\d+)$')
SITES_RE = re.compile(r'Sites=<([^>]*)>')

def make_windows(seq: str, w: int = WINDOW_SIZE, s: int = STRIDE) -> List[Tuple[int, str]]:
    L = len(seq)
    if L <= w:
        return [(0, seq)]
    starts = list(range(0, L - w + 1, s))
    if starts[-1] + w < L:
        starts.append(L - w)
    return [(i, seq[i:i + w]) for i in starts]

def build_prompt(window_seq: str) -> str:
    return PROMPT_TEMPLATE.format(instruction=INSTRUCTION, input=f'Seq=<{window_seq}>')

def extract_sites(text: str) -> List[Tuple[str, int]]:
    m = SITES_RE.search(text)
    if not m:
        return []
    body = m.group(1).strip()
    if body == '':
        return []
    out = []
    for part in body.split(','):
        mm = SITE_RE.match(part.strip())
        if mm:
            out.append((mm.group(1), int(mm.group(2))))
    return out

@torch.no_grad()
def generate_batch(prompts: List[str]) -> List[str]:
    enc = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[:, enc.input_ids.shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

_t = 'CQIVLTPELEGVEFALPKITR'
print('Sample generation:')
print(' input  :', _t)
print(' output :', generate_batch([build_prompt(_t)])[0].strip())

## 5. Calibration: Deriving the F1-Optimal Consensus Threshold

Procedure for each calibration protein:

1. The rows of `calibration.csv` for that protein define the set of windows on which to run inference (windowing convention is identical to training: 21-mer, stride 5).
2. Greedy generation is performed on each window in batches. Predicted sites are parsed, and any prediction whose residue letter does not match the window sequence at the indicated position is discarded.
3. Each validated prediction is mapped to its 1-indexed full-protein position, incrementing `predicted[i]`. `covered[i]` is incremented for every residue spanned by any window.
4. `y_score[i] = predicted[i] / covered[i]` for residues covered by at least one window. `y_true[i]` is read from the `whole_protein_binary_mask` column (1 = PTM site, 0 = non-site).
5. The per-residue arrays are flattened across all calibration proteins (only covered residues contribute), and the F1-maximizing threshold over the flattened arrays is selected as the decision threshold.

In [ ]:
cal_df = pd.read_csv(CALIBRATION_CSV)
cal_df['seq_sequence'] = cal_df['seq_sequence'].astype(str).str.strip().str.upper()
print(f'Calibration windows: {len(cal_df):,}')
print(f'Unique proteins:     {cal_df["Uniprot_ID"].nunique():,}')
print(f'has_ptm counts:      {cal_df["has_ptm"].value_counts().to_dict()}')
cal_df[['Uniprot_ID', 'window_start', 'seq_sequence', 'sites_in_window']].head()

In [ ]:
cal_proteins: Dict[str, dict] = {}

for row in cal_df.itertuples(index=False):
    uid = row.Uniprot_ID
    L = int(row.seq_length)
    if uid not in cal_proteins:
        mask = str(row.whole_protein_binary_mask)
        y_true = np.zeros(L, dtype=np.int8)
        for i, c in enumerate(mask[:L]):
            if c == '1':
                y_true[i] = 1
        cal_proteins[uid] = {
            'length': L,
            'seq_chars': ['?'] * L,
            'y_true': y_true,
            'covered': np.zeros(L, dtype=np.int32),
            'predicted': np.zeros(L, dtype=np.int32),
            'windows': [],
        }
    start_0 = int(row.window_start) - 1
    w_seq = str(row.seq_sequence)
    cal_proteins[uid]['windows'].append((start_0, w_seq))
    for j, ch in enumerate(w_seq):
        pos = start_0 + j
        if 0 <= pos < L:
            cal_proteins[uid]['seq_chars'][pos] = ch

for uid, p in cal_proteins.items():
    p['seq'] = ''.join(p['seq_chars'])

total_windows = sum(len(p['windows']) for p in cal_proteins.values())
print(f'Built per-protein structure: {len(cal_proteins)} proteins, {total_windows:,} windows.')

In [ ]:
cal_window_jobs = []
for uid, p in cal_proteins.items():
    for w_idx, (start, w_seq) in enumerate(p['windows']):
        cal_window_jobs.append((uid, w_idx, start, w_seq))

print(f'Running inference on {len(cal_window_jobs):,} calibration windows...')
cal_completions = [None] * len(cal_window_jobs)
t0 = time.time()
for i in tqdm(range(0, len(cal_window_jobs), BATCH_SIZE), desc='Calibration inference'):
    batch = cal_window_jobs[i:i + BATCH_SIZE]
    prompts = [build_prompt(w_seq) for _, _, _, w_seq in batch]
    outs = generate_batch(prompts)
    for j, c in enumerate(outs):
        cal_completions[i + j] = c
print(f'Done in {time.time() - t0:.1f}s.')

In [ ]:
for (uid, w_idx, start, w_seq), comp in zip(cal_window_jobs, cal_completions):
    p = cal_proteins[uid]
    L = p['length']
    end = min(start + len(w_seq), L)
    p['covered'][start:end] += 1
    for letter, pos_local in extract_sites(comp):
        if not (1 <= pos_local <= len(w_seq)):
            continue
        pos_full_0 = start + pos_local - 1
        if not (0 <= pos_full_0 < L):
            continue
        if p['seq'][pos_full_0] != letter:
            continue
        p['predicted'][pos_full_0] += 1

for p in cal_proteins.values():
    mask = p['covered'] > 0
    p['mask'] = mask
    p['y_score'] = np.zeros(p['length'], dtype=np.float32)
    p['y_score'][mask] = p['predicted'][mask] / p['covered'][mask]

cal_y_true  = np.concatenate([p['y_true'][p['mask']]  for p in cal_proteins.values()]).astype(np.int8)
cal_y_score = np.concatenate([p['y_score'][p['mask']] for p in cal_proteins.values()]).astype(np.float32)
cal_residues = np.concatenate([
    np.frombuffer(p['seq'].encode('ascii'), dtype=np.uint8)[p['mask']] for p in cal_proteins.values()
])

print(f'Calibration residues evaluated: {len(cal_y_true):,}')
print(f'Positive prevalence (cal): {cal_y_true.mean() * 100:.2f}%')
print(f'Mean predicted score (cal): {cal_y_score.mean():.4f}')

In [ ]:
cal_fpr, cal_tpr, cal_thresh = roc_curve(cal_y_true, cal_y_score)
cal_auc = roc_auc_score(cal_y_true, cal_y_score)

candidates = np.unique(cal_y_score)
if len(candidates) > 500:
    candidates = np.linspace(0, 1, 501)

def f1_for(y_true, y_score, t):
    yp = (y_score >= t).astype(np.int8)
    p, r, f, _ = precision_recall_fscore_support(y_true, yp, average='binary', zero_division=0)
    return f, p, r

sweep = [(float(t), *f1_for(cal_y_true, cal_y_score, t)) for t in candidates]
best_idx = int(np.argmax([s[1] for s in sweep]))
best_t, best_f1, best_p, best_r = sweep[best_idx]
LOCKED_THRESHOLD = float(best_t)

print('=' * 60)
print(f'Calibration ROC AUC      : {cal_auc:.4f}')
print(f'F1-optimal threshold     : {LOCKED_THRESHOLD:.4f}')
print(f'  calibration precision  : {best_p:.4f}')
print(f'  calibration recall     : {best_r:.4f}')
print(f'  calibration F1         : {best_f1:.4f}')
print('=' * 60)

In [ ]:
ts, f1s, ps, rs = zip(*sweep)
fig, (ax_roc, ax_sweep) = plt.subplots(1, 2, figsize=(13, 5))

ax_roc.plot(cal_fpr, cal_tpr, label=f'Calibration ROC (AUC={cal_auc:.3f})', linewidth=2)
ax_roc.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
ax_roc.set_xlabel('False positive rate')
ax_roc.set_ylabel('True positive rate')
ax_roc.set_title('Calibration ROC')
ax_roc.legend(loc='lower right')
ax_roc.grid(True, alpha=0.3)

ax_sweep.plot(ts, f1s, label='F1', linewidth=2)
ax_sweep.plot(ts, ps, label='Precision', alpha=0.7)
ax_sweep.plot(ts, rs, label='Recall', alpha=0.7)
ax_sweep.axvline(LOCKED_THRESHOLD, linestyle='--', color='red', alpha=0.7,
                 label=f'Locked t = {LOCKED_THRESHOLD:.3f}')
ax_sweep.set_xlabel('Consensus threshold')
ax_sweep.set_ylabel('Metric')
ax_sweep.set_title('Calibration threshold sweep')
ax_sweep.legend()
ax_sweep.grid(True, alpha=0.3)

fig.tight_layout()
cal_plot_path = os.path.join(EVAL_DIR, 'calibration_threshold_sweep.png')
fig.savefig(cal_plot_path, dpi=150)
plt.show()
print('Saved:', cal_plot_path)

## 6. Test Set Evaluation

`test.csv` contains full proteins (not pre-windowed). For each protein, a window of size 21 with stride 5 is slid across the sequence, inference is run on each window, predictions are parsed, and per-residue consensus scores are aggregated. The threshold derived from the calibration set in §5 is then applied to produce the final binary site calls. No threshold selection is performed on the test set; the metrics reported in §7 are unbiased point estimates of generalization.

In [ ]:
df_test = pd.read_csv(TEST_CSV)
df_test = df_test.dropna(subset=['seq_sequence']).copy()
df_test['seq_sequence'] = df_test['seq_sequence'].str.strip().str.upper()
df_test = df_test[df_test['seq_sequence'].str.fullmatch(r'[A-Z]+')].copy()

def parse_sites(s):
    if pd.isna(s) or str(s).strip() == '':
        return []
    out = []
    for part in str(s).split(','):
        m = SITE_RE.match(part.strip())
        if m:
            out.append((m.group(1), int(m.group(2))))
    return out

df_test['sites'] = df_test['Combined_Sites'].apply(parse_sites)

def validate_sites(row):
    seq, L = row['seq_sequence'], len(row['seq_sequence'])
    return [(letter, pos) for letter, pos in row['sites']
            if 1 <= pos <= L and seq[pos - 1] == letter]

df_test['sites'] = df_test.apply(validate_sites, axis=1)
df_test = df_test.reset_index(drop=True)

print(f'Test proteins: {len(df_test):,}')
print(f'Avg length:   {df_test["seq_sequence"].str.len().mean():.1f}')
print(f'Max length:   {df_test["seq_sequence"].str.len().max()}')
print(f'Total ground-truth PTM sites: {sum(len(s) for s in df_test["sites"]):,}')
df_test[['Uniprot_ID', 'Combined_Sites', 'sites']].head()

In [ ]:
test_window_jobs = []
for prot_idx, row in df_test.iterrows():
    for start, w_seq in make_windows(row['seq_sequence']):
        test_window_jobs.append((prot_idx, start, w_seq))

print(f'Test windows to score: {len(test_window_jobs):,}')
print(f'Avg windows per protein: {len(test_window_jobs) / len(df_test):.1f}')

In [ ]:
test_completions = [None] * len(test_window_jobs)
t0 = time.time()
for i in tqdm(range(0, len(test_window_jobs), BATCH_SIZE), desc='Test inference'):
    batch = test_window_jobs[i:i + BATCH_SIZE]
    prompts = [build_prompt(w_seq) for _, _, w_seq in batch]
    outs = generate_batch(prompts)
    for j, c in enumerate(outs):
        test_completions[i + j] = c
print(f'Done in {time.time() - t0:.1f}s ({len(test_window_jobs) / (time.time() - t0):.1f} windows/s)')

In [ ]:
test_proteins = []
for prot_idx, row in df_test.iterrows():
    seq = row['seq_sequence']
    L = len(seq)
    p = {
        'uniprot_id': row['Uniprot_ID'],
        'seq': seq,
        'length': L,
        'covered': np.zeros(L, dtype=np.int32),
        'predicted': np.zeros(L, dtype=np.int32),
        'y_true': np.zeros(L, dtype=np.int8),
    }
    for _letter, pos in row['sites']:
        p['y_true'][pos - 1] = 1
    test_proteins.append(p)

for (prot_idx, start, w_seq), comp in zip(test_window_jobs, test_completions):
    p = test_proteins[prot_idx]
    L = p['length']
    end = min(start + len(w_seq), L)
    p['covered'][start:end] += 1
    for letter, pos_local in extract_sites(comp):
        if not (1 <= pos_local <= len(w_seq)):
            continue
        pos_full_0 = start + pos_local - 1
        if not (0 <= pos_full_0 < L):
            continue
        if p['seq'][pos_full_0] != letter:
            continue
        p['predicted'][pos_full_0] += 1

for p in test_proteins:
    mask = p['covered'] > 0
    p['mask'] = mask
    p['y_score'] = np.zeros(p['length'], dtype=np.float32)
    p['y_score'][mask] = p['predicted'][mask] / p['covered'][mask]

test_y_true  = np.concatenate([p['y_true'][p['mask']]  for p in test_proteins]).astype(np.int8)
test_y_score = np.concatenate([p['y_score'][p['mask']] for p in test_proteins]).astype(np.float32)
test_residues = np.concatenate([
    np.frombuffer(p['seq'].encode('ascii'), dtype=np.uint8)[p['mask']] for p in test_proteins
])

print(f'Test residues evaluated: {len(test_y_true):,}')
print(f'Positive prevalence (test): {test_y_true.mean() * 100:.2f}%')

## 7. Final Test Metrics

The reported metrics are Accuracy, Precision, Recall (sensitivity), Specificity, and F1, all evaluated at the calibration-derived threshold.

ROC AUC on the test set is also reported. It is threshold-free and complements the point metrics by characterizing the underlying ranking ability of the model independently of the chosen operating point.

In [ ]:
test_y_pred = (test_y_score >= LOCKED_THRESHOLD).astype(np.int8)
cm = confusion_matrix(test_y_true, test_y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

accuracy    = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0
precision   = tp / (tp + fp) if (tp + fp) else 0.0
recall      = tp / (tp + fn) if (tp + fn) else 0.0
specificity = tn / (tn + fp) if (tn + fp) else 0.0
f1          = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

try:
    test_auc = roc_auc_score(test_y_true, test_y_score)
except ValueError:
    test_auc = float('nan')

print('=' * 60)
print(f'FINAL TEST METRICS  (locked threshold t = {LOCKED_THRESHOLD:.4f})')
print('=' * 60)
print(f'  Accuracy    : {accuracy:.4f}')
print(f'  Precision   : {precision:.4f}')
print(f'  Recall      : {recall:.4f}  (sensitivity)')
print(f'  Specificity : {specificity:.4f}')
print(f'  F1          : {f1:.4f}')
print(f'  ROC AUC     : {test_auc:.4f}  (threshold-free)')
print()
print(f'  Confusion matrix:')
print(f'    TN = {tn:>10,}    FP = {fp:>10,}')
print(f'    FN = {fn:>10,}    TP = {tp:>10,}')
print('=' * 60)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(
    cm, annot=True, fmt=',d', cmap='Blues', cbar=False, ax=ax,
    xticklabels=['Pred: non-site', 'Pred: site'],
    yticklabels=['True: non-site', 'True: site'],
)
ax.set_title(f'Test confusion matrix at locked t = {LOCKED_THRESHOLD:.4f}')
fig.tight_layout()
cm_path = os.path.join(EVAL_DIR, 'confusion_matrix.png')
fig.savefig(cm_path, dpi=150)
plt.show()
print('Saved:', cm_path)

In [ ]:
test_fpr, test_tpr, _ = roc_curve(test_y_true, test_y_score)
op_fpr = 1.0 - specificity
op_tpr = recall

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(test_fpr, test_tpr, label=f'Test ROC (AUC = {test_auc:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5, label='Random')
ax.scatter([op_fpr], [op_tpr], color='red', s=80, zorder=3,
           label=f'Locked operating point\n(t = {LOCKED_THRESHOLD:.3f})')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('Test ROC with locked operating point')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
roc_path = os.path.join(EVAL_DIR, 'roc_curve.png')
fig.savefig(roc_path, dpi=150)
plt.show()
print('Saved:', roc_path)

## 8. Per-Residue-Type Breakdown

ADPr preferentially targets a small set of residues (D, E, R, K, S). The table below restricts each metric to positions of a single residue type, at the calibration-derived threshold.

In [ ]:
per_type = []
for aa in ADPR_RESIDUES:
    mask = test_residues == ord(aa)
    if mask.sum() == 0:
        continue
    yt = test_y_true[mask]
    ys = test_y_score[mask]
    yp = (ys >= LOCKED_THRESHOLD).astype(np.int8)
    cm_aa = confusion_matrix(yt, yp, labels=[0, 1])
    tn_a, fp_a, fn_a, tp_a = cm_aa.ravel()
    acc_a = (tp_a + tn_a) / max(tp_a + tn_a + fp_a + fn_a, 1)
    p_a = tp_a / max(tp_a + fp_a, 1) if (tp_a + fp_a) else 0.0
    r_a = tp_a / max(tp_a + fn_a, 1) if (tp_a + fn_a) else 0.0
    s_a = tn_a / max(tn_a + fp_a, 1) if (tn_a + fp_a) else 0.0
    f_a = 2 * p_a * r_a / (p_a + r_a) if (p_a + r_a) else 0.0
    try:
        auc_a = roc_auc_score(yt, ys) if yt.sum() > 0 else float('nan')
    except ValueError:
        auc_a = float('nan')
    per_type.append({
        'residue': aa,
        'n_residues': int(mask.sum()),
        'n_positive': int(yt.sum()),
        'auc': auc_a,
        'accuracy': acc_a,
        'precision': p_a,
        'recall': r_a,
        'specificity': s_a,
        'f1': f_a,
    })
per_type_df = pd.DataFrame(per_type)
per_type_df

## 9. Persist Artifacts

Three artifacts are written to disk and uploaded to the Hugging Face repository:

1. **`inference_config.json`** — the inference-time configuration: window size, stride, calibration-derived threshold, instruction string, and maximum new tokens. Downstream inference code loads this file directly.
2. **`metrics_summary.json`** — the complete numerical results from this run, including the calibration AUC and the selected threshold for reproducibility.
3. **`model_eval_results.csv`** — per-protein test predictions at the calibration-derived threshold.

In [ ]:
inference_config = {
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'consensus_threshold': float(LOCKED_THRESHOLD),
    'max_new_tokens': MAX_NEW_TOKENS,
    'instruction': INSTRUCTION,
    'prompt_template': PROMPT_TEMPLATE,
    'output_format': 'Sites=<X1,X2,...> where Xi is one-letter residue + 1-indexed position within the window',
}
inf_cfg_path = os.path.join(EVAL_DIR, 'inference_config.json')
with open(inf_cfg_path, 'w') as f:
    json.dump(inference_config, f, indent=2)
print('Saved:', inf_cfg_path)
print(json.dumps(inference_config, indent=2))

In [ ]:
summary = {
    'model_repo': HF_REPO_ID,
    'calibration_csv': CALIBRATION_CSV,
    'test_csv': TEST_CSV,
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
    'locked_threshold': float(LOCKED_THRESHOLD),
    'calibration': {
        'n_proteins': int(len(cal_proteins)),
        'n_windows': int(len(cal_window_jobs)),
        'n_residues_evaluated': int(len(cal_y_true)),
        'positive_prevalence': float(cal_y_true.mean()),
        'roc_auc': float(cal_auc),
        'f1_at_locked_threshold': float(best_f1),
        'precision_at_locked_threshold': float(best_p),
        'recall_at_locked_threshold': float(best_r),
    },
    'test': {
        'n_proteins': int(len(df_test)),
        'n_windows': int(len(test_window_jobs)),
        'n_residues_evaluated': int(len(test_y_true)),
        'positive_prevalence': float(test_y_true.mean()),
        'roc_auc': float(test_auc) if not math.isnan(test_auc) else None,
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'specificity': float(specificity),
        'f1': float(f1),
        'confusion_matrix': {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)},
    },
    'per_residue_type': per_type_df.to_dict(orient='records'),
}
summary_path = os.path.join(EVAL_DIR, 'metrics_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', summary_path)

In [ ]:
rows = []
for p, row in zip(test_proteins, df_test.itertuples(index=False)):
    L = p['length']
    pred_sites = [(p['seq'][i], i + 1) for i in range(L)
                  if p['mask'][i] and p['y_score'][i] >= LOCKED_THRESHOLD]
    rows.append({
        'Uniprot_ID': p['uniprot_id'],
        'length': L,
        'true_sites': ', '.join(f'{a}{pos}' for a, pos in row.sites),
        'predicted_sites': ', '.join(f'{a}{pos}' for a, pos in pred_sites),
        'n_true': len(row.sites),
        'n_pred': len(pred_sites),
    })
pred_df = pd.DataFrame(rows)
pred_csv = os.path.join(EVAL_DIR, 'model_eval_results.csv')
pred_df.to_csv(pred_csv, index=False)
print('Saved:', pred_csv)
pred_df.head()

## 10. Generate and Push the Model Card

A `README.md` is generated documenting the inference architecture, the calibration and test methodology, and the full metrics. It includes a self-contained reference implementation of the inference pipeline.

In [ ]:
def pct(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return '—'
    return f'{x * 100:.2f}%'

def auc_str(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return '—'
    return f'{x:.3f}'

per_type_rows = '\n'.join(
    f"| {r['residue']} | {r['n_residues']:,} | {r['n_positive']:,} | {auc_str(r['auc'])} "
    f"| {pct(r['accuracy'])} | {pct(r['precision'])} | {pct(r['recall'])} | {pct(r['specificity'])} | {pct(r['f1'])} |"
    for r in per_type
)
per_type_table_md = (
    '| Residue | Total | Positive | AUC | Accuracy | Precision | Recall | Specificity | F1 |\n'
    '|---|---|---|---|---|---|---|---|---|\n'
    + per_type_rows
)

card = f'''---
base_model: GreatCaptainNemo/ProLLaMA_Stage_1
tags:
  - protein
  - ptm
  - adp-ribosylation
  - lora
  - peft
library_name: transformers
pipeline_tag: text-generation
---

# ADPr-LLaMA

LoRA fine-tune of [`GreatCaptainNemo/ProLLaMA_Stage_1`](https://huggingface.co/GreatCaptainNemo/ProLLaMA_Stage_1) for predicting ADP-ribosylation (ADPr) post-translational-modification (PTM) sites in protein sequences.

## Task

Given a 21-residue peptide window, the model generates the list of ADPr-modified positions in the format `Sites=<R5,D12,...>` (residue letter + 1-indexed position within the window).

## Inference architecture

Full-protein inference proceeds in three steps:

1. **Sliding windows.** A 21-residue window is slid across the input sequence with stride 5; a tail window is appended so the final residues are covered.
2. **Per-window generation.** The model generates `Sites=<...>` for each window. Each predicted residue letter is validated against the window sequence at the indicated position; mismatches are discarded. Validated predictions are mapped to full-protein coordinates and accumulated into a per-residue **consensus score**, defined as `(# windows predicting the residue as a site) / (# windows covering the residue)`.
3. **Thresholding.** Residues whose consensus score is at least **t = {LOCKED_THRESHOLD:.4f}** are reported as predicted sites. This threshold is the F1-optimal value selected from the per-residue ROC on a held-out calibration set.

The threshold, windowing parameters, and prompt template are persisted in [`inference_config.json`](./inference_config.json) and can be loaded directly by downstream inference code.

### Reference implementation

```python
import re, json, torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

REPO = "{HF_REPO_ID}"

cfg = json.load(open(hf_hub_download(REPO, "inference_config.json")))
tok = AutoTokenizer.from_pretrained(REPO)
if tok.pad_token is None:
    tok.pad_token = tok.unk_token
mdl = AutoModelForCausalLM.from_pretrained(
    REPO, torch_dtype=torch.float16, device_map="auto"
).eval()

SITE_RE  = re.compile(r"^([A-Z])(\\d+)$")
SITES_RE = re.compile(r"Sites=<([^>]*)>")

@torch.no_grad()
def predict_sites(seq: str):
    """Predict ADPr-site positions in a full protein.
    Returns sites like ['R161', 'D203'] in 1-indexed full-protein coords."""
    w, s, t = cfg["window_size"], cfg["stride"], cfg["consensus_threshold"]
    L = len(seq)

    # 1) Sliding windows with tail.
    if L <= w:
        starts = [0]
    else:
        starts = list(range(0, L - w + 1, s))
        if starts[-1] + w < L:
            starts.append(L - w)

    covered = [0] * L
    predicted = [0] * L

    for st in starts:
        win = seq[st:st + w]
        prompt = cfg["prompt_template"].format(
            instruction=cfg["instruction"], input=f"Seq=<{{win}}>"
        )
        enc = tok(prompt, return_tensors="pt").to(mdl.device)
        out = mdl.generate(
            **enc, max_new_tokens=cfg["max_new_tokens"], do_sample=False,
            pad_token_id=tok.pad_token_id,
        )
        text = tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

        # 2) Coverage + parsed predictions.
        for i in range(st, min(st + w, L)):
            covered[i] += 1
        m = SITES_RE.search(text)
        if not m:
            continue
        for part in m.group(1).split(","):
            mm = SITE_RE.match(part.strip())
            if not mm:
                continue
            letter, pos_local = mm.group(1), int(mm.group(2))
            pos_full = st + pos_local  # 1-indexed full-protein position
            if 1 <= pos_full <= L and seq[pos_full - 1] == letter:
                predicted[pos_full - 1] += 1

    # 3) Consensus + locked threshold.
    return [f"{{seq[i]}}{{i + 1}}" for i in range(L)
            if covered[i] > 0 and predicted[i] / covered[i] >= t]

sites = predict_sites("MASDEGKLFVGGLSFDTNEQALEQVFSKYGQISEVVVVKDRETQRSRGFGFVTFENIDDAKDAMMAMNGK")
print(sites)
```

## Training

- **Base model:** `GreatCaptainNemo/ProLLaMA_Stage_1`
- **Method:** LoRA SFT via `trl.SFTTrainer` + `peft.LoraConfig`
- **LoRA config:** r=64, alpha=128, dropout=0.05, target modules = q,k,v,o,gate,down,up_proj
- **Optimizer:** AdamW, lr=3e-4, cosine schedule, warmup=40 steps, max_grad_norm=1.0
- **Batching:** per-device batch 16 × grad_accum 8 (effective 128) at bf16, max_seq_length 2048
- **Epochs:** up to 8, with `EarlyStoppingCallback(patience=3)` on `eval_loss` and `load_best_model_at_end=True`
- **Train/val split:** group-shuffle by `Uniprot_ID` (10% val) — ensures no protein appears in both splits
- **Training distribution:** positives-only (`has_ptm == 1`), following the precedent established by prior state-of-the-art ADPr-site prediction work; without this filtering, models in this family fail to learn site localization. The resulting tendency to over-predict is addressed at inference by the consensus-and-threshold mechanism described above.

![training_loss](./training_loss.png)

## Evaluation methodology

Two disjoint held-out sets are used to separate threshold selection from final metric reporting:

- **Calibration set** ({int(len(cal_proteins))} proteins held out from the training data, {int(len(cal_y_true)):,} per-residue observations): the F1-maximizing threshold over the per-residue consensus ROC is selected from this set.
- **Test set** ({int(len(df_test))} unseen full proteins, {int(len(test_y_true)):,} per-residue observations): the calibration-derived threshold is applied without modification; the metrics below are unbiased point estimates of generalization.

### Calibration results

- ROC AUC: **{cal_auc:.4f}**
- Selected threshold (F1-optimal on calibration): **t = {LOCKED_THRESHOLD:.4f}**
- Calibration metrics at this threshold: precision {pct(best_p)}, recall {pct(best_r)}, F1 {pct(best_f1)}

![calibration_threshold_sweep](./calibration_threshold_sweep.png)

### Final test metrics

| Metric | Value |
|---|---|
| Accuracy | {pct(accuracy)} |
| Precision | {pct(precision)} |
| Recall (sensitivity) | {pct(recall)} |
| Specificity | {pct(specificity)} |
| F1 | {pct(f1)} |
| ROC AUC (threshold-free) | {auc_str(test_auc)} |

**Confusion matrix:**

|              | Pred: non-site | Pred: site |
|---|---|---|
| **True: non-site** | {tn:,} | {fp:,} |
| **True: site**     | {fn:,} | {tp:,} |

![confusion_matrix](./confusion_matrix.png)

![roc_curve](./roc_curve.png)

### Per-residue-type breakdown on test (D, E, R, K, S are the canonical ADPr targets)

{per_type_table_md}

## Limitations

- **Window-local outputs.** The model emits positions inside a 21-residue window. Full-protein predictions are produced by the sliding-window aggregation described in *Inference architecture*; very short proteins (`length < 21`) are scored as a single window.
- **No structural context.** The model sees only primary sequence; structurally-disfavored false positives cannot be filtered without external 3D information.
- **Dataset coverage.** Training derives from curated ADPriboDB-style sources; performance on de-novo proteins or non-canonical residues is unknown.

## Reproduction

Execute `training/train_adpr_llama.ipynb` followed by `evaluation/evaluate_adpr_llama.ipynb` from the source repository. Both notebooks are self-contained and intended for execution in Google Colab.
'''

card_path = os.path.join(EVAL_DIR, 'README.md')
with open(card_path, 'w') as f:
    f.write(card)
print('Wrote', card_path)
print('First 1500 chars:\n')
print(card[:1500])

In [ ]:
if PUSH_MODEL_CARD:
    eval_artifacts = [
        'README.md',
        'inference_config.json',
        'metrics_summary.json',
        'model_eval_results.csv',
        'roc_curve.png',
        'confusion_matrix.png',
        'calibration_threshold_sweep.png',
    ]
    api = HfApi()
    api.upload_folder(
        folder_path=EVAL_DIR,
        repo_id=HF_REPO_ID,
        token=HF_TOKEN,
        allow_patterns=eval_artifacts,
        delete_patterns=eval_artifacts,
        commit_message='Push evaluation artifacts and model card',
    )
    print('Model card pushed to', f'https://huggingface.co/{HF_REPO_ID}')
else:
    print('PUSH_MODEL_CARD=False; skipped Hub upload.')

## Output Artifacts

The `eval_outputs/` directory contains:
- `inference_config.json` — inference-time configuration (window size, stride, threshold, prompt template)
- `metrics_summary.json` — full numerical results from this run
- `model_eval_results.csv` — per-protein predicted vs. true sites on the test set
- `calibration_threshold_sweep.png`, `roc_curve.png`, `confusion_matrix.png` — figures
- `README.md` — model card uploaded to the Hugging Face repository